# 04 Bin Ablation

This notebook tests range-bin crops in memory, without saving processed arrays, models, predictions, or summary files.


In [1]:
!pip -q install huggingface_hub


# Setup


In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

from pathlib import Path

if Path('/content/drive/MyDrive').exists():
    PROJECT_DIR = Path('/content/drive/MyDrive/Multi-Person-Detection')
elif Path.cwd().name == 'notebooks':
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

RAW_DATA_DIR = PROJECT_DIR / 'data' / 'raw'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Project directory:', PROJECT_DIR)
print('Raw data directory:', RAW_DATA_DIR)


Mounted at /content/drive
Project directory: /content/drive/MyDrive/Multi-Person-Detection
Raw data directory: /content/drive/MyDrive/Multi-Person-Detection/data/raw


# Imports and Configuration


In [3]:
"""Run simple in-memory range-bin ablation experiments."""

from __future__ import annotations

import argparse
import gc
import os
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import tensorflow as tf
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
from tensorflow import keras
from tensorflow.keras import layers

try:
    from huggingface_hub import snapshot_download
except ImportError as exc:
    raise SystemExit(
        "Missing dependency: huggingface_hub. Install it with:\n"
        "  pip install huggingface_hub"
    ) from exc


RANDOM_STATE = 42

SEQ_LEN = 16
FRAME_STRIDE = 16
USE_LOG = True

DECLUTTER_MODE = "ema"
EMA_ALPHA = 0.98

USE_MAG = False
USE_DECLUTTERED_MAG = True
USE_MOTION_MAG = True
USE_PHASE_DIFF = True
USE_LOCAL_VARIANCE = True
VARIANCE_WINDOW = 9

ROOM_X = 4.8
ROOM_Y = 7.2
MAX_PEOPLE = 4
NORMALIZATION_EPS = 1e-6
SELECTED_CHANNELS = [0, 4, 7, 11, 12, 15]

HEATMAP_H = 24
HEATMAP_W = 16
HEATMAP_SIGMA = 0.75

MATCH_THRESHOLD = 1.0
NMS_SIZE = 3

VALIDATION_WINDOWS_BY_PEOPLE_COUNT = {
    0: 0,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
}
TEST_WINDOWS_BY_PEOPLE_COUNT = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
}


@dataclass(frozen=True)
class BinConfig:
    start: int
    end: int

    @property
    def name(self) -> str:
        return f"bins_{self.start}_{self.end}"

    @property
    def width(self) -> int:
        return self.end - self.start


BIN_CONFIGS = [BinConfig(start, end) for start in (4, 5, 6) for end in range(50, 61)]


def parse_bin_config(value: str) -> BinConfig:
    normalized = value.replace(",", ":")
    parts = normalized.split(":")
    if len(parts) != 2:
        raise argparse.ArgumentTypeError(
            f"Expected START:END, got {value!r}."
        )
    start, end = (int(parts[0]), int(parts[1]))
    if not 0 <= start < end <= 120:
        raise argparse.ArgumentTypeError(
            f"Invalid bin range [{start}:{end}]. Expected 0 <= start < end <= 120."
        )
    return BinConfig(start=start, end=end)


def resolve_project_dir(cli_project_dir: str | None) -> Path:
    if cli_project_dir:
        return Path(cli_project_dir).expanduser().resolve()

    colab_project = Path("/content/drive/MyDrive/Multi-Person-Detection")
    if colab_project.exists():
        return colab_project

    cwd = Path.cwd().resolve()
    if cwd.name == "scripts":
        return cwd.parent
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


def find_dataset(project_dir: Path, dataset_repo: str, hf_token: str | None = None) -> list[str]:
    local_dataset_dir = project_dir / "data" / "raw" / "multi-person-localization"
    if any(local_dataset_dir.rglob("*.npz")):
        dataset_path = local_dataset_dir
    else:
        try:
            dataset_path = Path(
                snapshot_download(
                    repo_id=dataset_repo,
                    repo_type="dataset",
                    token=hf_token,
                )
            )
        except Exception as exc:
            raise SystemExit(
                "\nCould not download the Hugging Face dataset.\n\n"
                f"Dataset repo: {dataset_repo}\n\n"
                "If the dataset is private or gated, authenticate first in Colab:\n\n"
                "  from huggingface_hub import login\n"
                "  login()\n\n"
                "or save a Colab Secret named HF_TOKEN and rerun the launcher notebook.\n\n"
                "Alternative: place the .npz files under:\n\n"
                f"  {local_dataset_dir}\n\n"
                "The script will use local .npz files from that folder without calling Hugging Face.\n"
            ) from exc

    npz_files = sorted(str(path) for path in dataset_path.rglob("*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No .npz files found under {dataset_path}.")

    print(f"Dataset path: {dataset_path}")
    print(f"Total .npz files: {len(npz_files)}")
    return npz_files


def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)



# Experiment Settings


In [4]:
DATASET_REPO = 'HAEEAI/multi-person-localization'
HF_TOKEN = None  # or set this manually if needed

EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 1e-4

# Optional smoke-test limits. Keep None for the real ablation.
MAX_FILES_PER_SPLIT = None
MAX_TRAIN_SAMPLES = None
MAX_VALIDATION_SAMPLES = None
MAX_TEST_SAMPLES = None

# Default grid: bin_start in {4, 5, 6}, bin_end from 50 to 60.
BIN_CONFIGS


[BinConfig(start=4, end=50),
 BinConfig(start=4, end=51),
 BinConfig(start=4, end=52),
 BinConfig(start=4, end=53),
 BinConfig(start=4, end=54),
 BinConfig(start=4, end=55),
 BinConfig(start=4, end=56),
 BinConfig(start=4, end=57),
 BinConfig(start=4, end=58),
 BinConfig(start=4, end=59),
 BinConfig(start=4, end=60),
 BinConfig(start=5, end=50),
 BinConfig(start=5, end=51),
 BinConfig(start=5, end=52),
 BinConfig(start=5, end=53),
 BinConfig(start=5, end=54),
 BinConfig(start=5, end=55),
 BinConfig(start=5, end=56),
 BinConfig(start=5, end=57),
 BinConfig(start=5, end=58),
 BinConfig(start=5, end=59),
 BinConfig(start=5, end=60),
 BinConfig(start=6, end=50),
 BinConfig(start=6, end=51),
 BinConfig(start=6, end=52),
 BinConfig(start=6, end=53),
 BinConfig(start=6, end=54),
 BinConfig(start=6, end=55),
 BinConfig(start=6, end=56),
 BinConfig(start=6, end=57),
 BinConfig(start=6, end=58),
 BinConfig(start=6, end=59),
 BinConfig(start=6, end=60)]

# Helper Functions


In [5]:
def compute_window_statistics(npz_files: list[str]) -> list[dict]:
    rows = []
    for npz_path in npz_files:
        with np.load(npz_path) as data:
            people_mask = data["people_mask"]
        people_per_frame = (people_mask > 0.5).sum(axis=1)
        people_count = int(people_per_frame[0])
        if not np.all(people_per_frame == people_count):
            raise ValueError(f"Variable people count in window: {npz_path}")
        rows.append(
            {
                "window": os.path.basename(npz_path),
                "total_frames": int(len(people_mask)),
                "people_count": people_count,
            }
        )
    return rows


def make_file_splits(npz_files: list[str], seed: int) -> tuple[list[str], list[str], list[str]]:
    stats = compute_window_statistics(npz_files)
    people_count_by_window = {
        row["window"]: row["people_count"]
        for row in stats
    }

    windows_by_people_count: dict[int, list[str]] = defaultdict(list)
    for npz_path in npz_files:
        people_count = people_count_by_window[os.path.basename(npz_path)]
        windows_by_people_count[people_count].append(npz_path)

    rng = np.random.default_rng(seed)
    for files in windows_by_people_count.values():
        files.sort()
        rng.shuffle(files)

    train_files: list[str] = []
    validation_files: list[str] = []
    test_files: list[str] = []

    for people_count in range(MAX_PEOPLE + 1):
        files = windows_by_people_count[people_count]
        validation_count = VALIDATION_WINDOWS_BY_PEOPLE_COUNT[people_count]
        test_count = TEST_WINDOWS_BY_PEOPLE_COUNT[people_count]
        reserved_count = validation_count + test_count

        if reserved_count >= len(files):
            raise ValueError(
                f"{people_count} people: reserve at least one window for training. "
                f"Available={len(files)}, validation={validation_count}, test={test_count}."
            )

        validation_files.extend(files[:validation_count])
        test_files.extend(files[validation_count:reserved_count])
        train_files.extend(files[reserved_count:])

    train_files = sorted(train_files)
    validation_files = sorted(validation_files)
    test_files = sorted(test_files)

    all_split_files = train_files + validation_files + test_files
    if len(all_split_files) != len(npz_files) or len(set(all_split_files)) != len(npz_files):
        raise ValueError("Train, validation, and test splits must cover every window exactly once.")

    print_split_summary("Train", train_files, people_count_by_window)
    print_split_summary("Validation", validation_files, people_count_by_window)
    print_split_summary("Test", test_files, people_count_by_window)

    return train_files, validation_files, test_files


def print_split_summary(
    split_name: str,
    files: list[str],
    people_count_by_window: dict[str, int],
) -> None:
    counts = {count: 0 for count in range(MAX_PEOPLE + 1)}
    for path in files:
        counts[people_count_by_window[os.path.basename(path)]] += 1
    print(f"{split_name}: {len(files)} windows | " + ", ".join(
        f"{count}p={num}" for count, num in counts.items()
    ))


def build_complex_cir(radar_iq: np.ndarray) -> np.ndarray:
    i = radar_iq[..., 0]
    q = radar_iq[..., 1]
    return (i + 1j * q).astype(np.complex64)


def apply_declutter(x: np.ndarray, mode: str = "mean", alpha: float = 0.98) -> np.ndarray:
    if mode == "mean":
        return x - x.mean(axis=0, keepdims=True)

    if mode == "ema":
        if not 0 <= alpha <= 1:
            raise ValueError("alpha must be in the range [0, 1].")

        output = np.empty_like(x)
        background = x[0].copy()
        output[0] = 0

        for t in range(1, len(x)):
            background = alpha * background + (1 - alpha) * x[t]
            output[t] = x[t] - background

        return output

    raise ValueError(f"Unknown declutter mode: {mode}")


def local_temporal_variance(x: np.ndarray, window: int = 9) -> np.ndarray:
    if window <= 0 or window % 2 == 0:
        raise ValueError("window must be a positive odd integer.")

    from numpy.lib.stride_tricks import sliding_window_view

    pad = window // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0), (0, 0)), mode="edge")
    windows = sliding_window_view(x_pad, window, axis=0)
    return windows.var(axis=-1, dtype=np.float32)


def build_radar_features(
    radar_iq: np.ndarray,
    bin_config: BinConfig,
) -> np.ndarray:
    number_of_frames = radar_iq.shape[0]
    number_of_bins = radar_iq.shape[3]

    radar_iq = radar_iq.transpose(0, 3, 1, 2, 4)
    radar_iq = radar_iq.reshape(number_of_frames, number_of_bins, -1, 2)
    radar_iq = radar_iq[:, :, SELECTED_CHANNELS]

    complex_cir = build_complex_cir(radar_iq)
    magnitude = np.abs(complex_cir).astype(np.float32)

    if USE_LOG:
        magnitude = np.log1p(magnitude)

    feature_blocks = []

    if USE_MAG:
        feature_blocks.append(magnitude)

    if USE_DECLUTTERED_MAG or USE_MOTION_MAG:
        decluttered_complex_cir = apply_declutter(
            complex_cir,
            DECLUTTER_MODE,
            EMA_ALPHA,
        )
        decluttered_magnitude = np.abs(decluttered_complex_cir).astype(np.float32)
        if USE_LOG:
            decluttered_magnitude = np.log1p(decluttered_magnitude)

    if USE_DECLUTTERED_MAG:
        feature_blocks.append(decluttered_magnitude)

    if USE_MOTION_MAG:
        motion_magnitude = np.abs(
            np.diff(
                decluttered_magnitude,
                axis=0,
                prepend=decluttered_magnitude[:1],
            )
        )
        feature_blocks.append(motion_magnitude)

    if USE_LOCAL_VARIANCE:
        feature_blocks.append(
            local_temporal_variance(magnitude, window=VARIANCE_WINDOW)
        )

    if USE_PHASE_DIFF:
        phase_difference = np.zeros_like(magnitude)
        phase_difference[1:] = np.angle(
            complex_cir[1:] * np.conj(complex_cir[:-1])
        )
        feature_blocks.append(phase_difference)

    if not feature_blocks:
        raise ValueError("Enable at least one feature block.")

    features = np.concatenate(feature_blocks, axis=2)
    return features[:, bin_config.start:bin_config.end, :].astype(np.float32)


def sort_people_by_x(positions: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    valid_positions = positions[mask.astype(bool)]
    valid_positions = valid_positions[np.argsort(valid_positions[:, 0])]
    number_of_people = min(len(valid_positions), MAX_PEOPLE)

    sorted_positions = np.zeros((MAX_PEOPLE, 2), dtype=np.float32)
    sorted_mask = np.zeros(MAX_PEOPLE, dtype=np.float32)
    sorted_positions[:number_of_people] = valid_positions[:number_of_people]
    sorted_mask[:number_of_people] = 1.0
    return sorted_positions, sorted_mask


def build_samples_from_window(
    npz_path: str,
    bin_config: BinConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    with np.load(npz_path) as data:
        radar = data["radar_cir_iq"].astype(np.float32, copy=False)
        people_positions = data["people_xy"].astype(np.float32, copy=False)
        people_masks = data["people_mask"].astype(np.float32, copy=False)

    features = build_radar_features(radar, bin_config)
    sequences = []
    position_labels = []
    mask_labels = []
    end_frames = []

    for end_frame in range(SEQ_LEN - 1, features.shape[0], FRAME_STRIDE):
        start_frame = end_frame - SEQ_LEN + 1
        positions, mask = sort_people_by_x(
            people_positions[end_frame],
            people_masks[end_frame],
        )
        sequences.append(features[start_frame:end_frame + 1])
        position_labels.append(positions)
        mask_labels.append(mask)
        end_frames.append(end_frame)

    if not sequences:
        number_of_features = len(SELECTED_CHANNELS) * enabled_feature_blocks()
        return (
            np.empty((0, SEQ_LEN, bin_config.width, number_of_features), dtype=np.float32),
            np.empty((0, MAX_PEOPLE, 2), dtype=np.float32),
            np.empty((0, MAX_PEOPLE), dtype=np.float32),
            np.empty((0,), dtype=np.int32),
        )

    return (
        np.stack(sequences).astype(np.float32),
        np.stack(position_labels).astype(np.float32),
        np.stack(mask_labels).astype(np.float32),
        np.array(end_frames, dtype=np.int32),
    )


def enabled_feature_blocks() -> int:
    return sum((
        USE_MAG,
        USE_DECLUTTERED_MAG,
        USE_MOTION_MAG,
        USE_PHASE_DIFF,
        USE_LOCAL_VARIANCE,
    ))


def build_dataset(
    files: list[str],
    bin_config: BinConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    radar_sequences_by_window = []
    position_labels_by_window = []
    mask_labels_by_window = []
    end_frame_indices_by_window = []
    sample_window_names = []

    for file_index, file_path in enumerate(files, start=1):
        print(f"  [{file_index}/{len(files)}] {Path(file_path).name}")
        sequences, positions, masks, end_frame_indices = build_samples_from_window(
            file_path,
            bin_config,
        )
        if len(sequences) == 0:
            continue

        radar_sequences_by_window.append(sequences)
        position_labels_by_window.append(positions)
        mask_labels_by_window.append(masks)
        end_frame_indices_by_window.append(end_frame_indices)
        sample_window_names.extend([Path(file_path).name] * len(sequences))

    if not radar_sequences_by_window:
        raise ValueError("No valid samples were generated.")

    return (
        np.concatenate(radar_sequences_by_window, axis=0),
        np.concatenate(position_labels_by_window, axis=0),
        np.concatenate(mask_labels_by_window, axis=0),
        np.concatenate(end_frame_indices_by_window, axis=0),
        np.array(sample_window_names),
    )


def sanitize_active_coordinates(
    split_name: str,
    y_coordinates: np.ndarray,
    y_mask: np.ndarray,
) -> np.ndarray:
    sanitized_coordinates = y_coordinates.copy()
    active = y_mask >= 0.5
    active_coordinates = sanitized_coordinates[active]
    if len(active_coordinates) == 0:
        print(f"{split_name}: no active coordinates to sanitize.")
        return sanitized_coordinates

    clipped_coordinates = np.clip(active_coordinates, [0.0, 0.0], [ROOM_X, ROOM_Y])
    changed = np.any(active_coordinates != clipped_coordinates, axis=1)
    sanitized_coordinates[active] = clipped_coordinates
    print(
        f"{split_name}: clipped {int(changed.sum())} active coordinates to "
        f"[0, {ROOM_X}] x [0, {ROOM_Y}]."
    )
    return sanitized_coordinates


def validate_dataset_split(
    split_name: str,
    x: np.ndarray,
    y_coordinates: np.ndarray,
    y_mask: np.ndarray,
    bin_config: BinConfig,
) -> None:
    expected_input_shape = (
        SEQ_LEN,
        bin_config.width,
        len(SELECTED_CHANNELS) * enabled_feature_blocks(),
    )
    if x.ndim != 4 or x.shape[1:] != expected_input_shape:
        raise ValueError(
            f"{split_name}: expected X shape (N, {expected_input_shape}), got {x.shape}."
        )
    if y_coordinates.shape != (len(x), MAX_PEOPLE, 2):
        raise ValueError(f"{split_name}: invalid coordinates shape {y_coordinates.shape}.")
    if y_mask.shape != (len(x), MAX_PEOPLE):
        raise ValueError(f"{split_name}: invalid mask shape {y_mask.shape}.")
    if not np.isfinite(x).all():
        raise ValueError(f"{split_name}: radar features contain NaN or infinite values.")
    if not np.isfinite(y_coordinates).all() or not np.isfinite(y_mask).all():
        raise ValueError(f"{split_name}: labels contain NaN or infinite values.")
    if not np.all((y_mask == 0.0) | (y_mask == 1.0)):
        raise ValueError(f"{split_name}: masks must contain only 0 and 1.")


def normalize_datasets(
    x_train: np.ndarray,
    x_validation: np.ndarray,
    x_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    mean = x_train.mean(axis=(0, 1, 2), dtype=np.float64, keepdims=True).astype(np.float32)
    std = x_train.std(axis=(0, 1, 2), dtype=np.float64, keepdims=True)
    std = np.maximum(std, NORMALIZATION_EPS).astype(np.float32)

    x_train = (x_train - mean) / std
    x_validation = (x_validation - mean) / std
    x_test = (x_test - mean) / std
    return x_train, x_validation, x_test, mean, std


def make_heatmap_from_points(
    xy: np.ndarray,
    mask: np.ndarray,
    h: int = HEATMAP_H,
    w: int = HEATMAP_W,
    sigma: float = HEATMAP_SIGMA,
) -> np.ndarray:
    heatmap = np.zeros((h, w), dtype=np.float32)
    ys = np.arange(h, dtype=np.float32)
    xs = np.arange(w, dtype=np.float32)
    yy, xx = np.meshgrid(ys, xs, indexing="ij")

    for i in range(len(mask)):
        if mask[i] < 0.5:
            continue

        px, py = xy[i]
        px = np.clip(px, 0.0, ROOM_X)
        py = np.clip(py, 0.0, ROOM_Y)
        col = (px / ROOM_X) * (w - 1)
        row = (py / ROOM_Y) * (h - 1)
        gaussian = np.exp(-((xx - col) ** 2 + (yy - row) ** 2) / (2 * sigma ** 2))
        heatmap = np.maximum(heatmap, gaussian)

    return heatmap[..., None]


def build_heatmaps(y_coordinates: np.ndarray, y_mask: np.ndarray) -> np.ndarray:
    return np.stack([
        make_heatmap_from_points(y_coordinates[i], y_mask[i])
        for i in range(len(y_coordinates))
    ]).astype(np.float32)


def conv_block(x, filters: int, kernel_size=(3, 3), name: str | None = None):
    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        name=None if name is None else name + "_conv1",
    )(x)
    x = layers.BatchNormalization(name=None if name is None else name + "_bn1")(x)
    x = layers.ReLU(name=None if name is None else name + "_relu1")(x)

    x = layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name=None if name is None else name + "_conv2",
    )(x)
    x = layers.BatchNormalization(name=None if name is None else name + "_bn2")(x)
    x = layers.ReLU(name=None if name is None else name + "_relu2")(x)
    return x


def resize_to_skip(x, skip, name: str):
    """Align decoder tensors for bin crops that are not divisible by pooling factors."""
    target_h = int(skip.shape[1])
    target_w = int(skip.shape[2])
    return layers.Resizing(
        target_h,
        target_w,
        interpolation="bilinear",
        name=name,
    )(x)


def build_radar_heatmap_unet(
    input_shape,
    heatmap_h: int = HEATMAP_H,
    heatmap_w: int = HEATMAP_W,
    prior_prob: float = 0.01,
):
    inputs = keras.Input(shape=input_shape, name="radar_input")

    c1 = conv_block(inputs, 32, kernel_size=(3, 7), name="enc1")
    p1 = layers.MaxPooling2D(pool_size=(1, 2), name="pool1")(c1)

    c2 = conv_block(p1, 64, kernel_size=(3, 5), name="enc2")
    p2 = layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(c2)

    c3 = conv_block(p2, 128, kernel_size=(3, 3), name="enc3")
    p3 = layers.MaxPooling2D(pool_size=(2, 2), name="pool3")(c3)

    b = conv_block(p3, 256, kernel_size=(3, 3), name="bottleneck")

    g = layers.GlobalAveragePooling2D(name="gap_context")(b)
    g = layers.Dense(256, activation="relu", name="context_dense1")(g)
    g = layers.Dense(256, activation="sigmoid", name="context_gate")(g)
    g = layers.Reshape((1, 1, 256), name="context_reshape")(g)
    b = layers.Multiply(name="context_modulation")([b, g])

    x = layers.UpSampling2D(size=(2, 2), interpolation="bilinear", name="up3")(b)
    x = resize_to_skip(x, c3, name="align3")
    x = layers.Concatenate(name="skip3")([x, c3])
    x = conv_block(x, 128, name="dec3")

    x = layers.UpSampling2D(size=(2, 2), interpolation="bilinear", name="up2")(x)
    x = resize_to_skip(x, c2, name="align2")
    x = layers.Concatenate(name="skip2")([x, c2])
    x = conv_block(x, 64, name="dec2")

    x = layers.UpSampling2D(size=(1, 2), interpolation="bilinear", name="up1")(x)
    x = resize_to_skip(x, c1, name="align1")
    x = layers.Concatenate(name="skip1")([x, c1])
    x = conv_block(x, 32, name="dec1")

    x = layers.Resizing(
        heatmap_h,
        heatmap_w,
        interpolation="bilinear",
        name="resize_to_heatmap",
    )(x)

    bias_init = tf.keras.initializers.Constant(
        -np.log((1.0 - prior_prob) / prior_prob)
    )
    outputs = layers.Conv2D(
        1,
        kernel_size=1,
        padding="same",
        activation="sigmoid",
        bias_initializer=bias_init,
        name="heatmap",
    )(x)

    return keras.Model(inputs, outputs, name="radar_heatmap_unet")


def stable_focal_dice_mass_loss(
    y_true,
    y_pred,
    alpha=0.90,
    gamma=2.0,
    dice_weight=0.3,
    mse_weight=1.0,
    mass_weight=0.20,
    center_weight=10.0,
    min_mass=5.0,
    eps=1e-6,
):
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

    bce = -(
        y_true * tf.math.log(y_pred)
        + (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )
    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
    focal = tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * bce)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3])
    dice = 1.0 - tf.reduce_mean((2.0 * intersection + eps) / (union + eps))

    weights = 1.0 + center_weight * y_true
    center_mse = tf.reduce_mean(weights * tf.square(y_true - y_pred))

    true_mass = tf.reduce_sum(y_true, axis=[1, 2, 3])
    pred_mass = tf.reduce_sum(y_pred, axis=[1, 2, 3])
    denom = tf.maximum(true_mass, min_mass)
    mass_error = tf.clip_by_value((pred_mass - true_mass) / denom, -5.0, 5.0)
    mass_loss = tf.reduce_mean(tf.square(mass_error))

    return focal + dice_weight * dice + mse_weight * center_mse + mass_weight * mass_loss


def heatmap_to_localizations(
    heatmap: np.ndarray,
    threshold: float,
    nms_size: int = NMS_SIZE,
    max_persons: int = MAX_PEOPLE,
) -> tuple[list[list[float]], list[float]]:
    if heatmap.ndim == 3:
        heatmap = heatmap[:, :, 0]

    h, w = heatmap.shape
    local_max = maximum_filter(heatmap, size=nms_size) == heatmap
    detected = local_max & (heatmap >= threshold)
    rows, cols = np.where(detected)

    if len(rows) == 0:
        return [], []

    scores = heatmap[rows, cols]
    order = np.argsort(scores)[::-1]
    rows = rows[order][:max_persons]
    cols = cols[order][:max_persons]
    scores = scores[order][:max_persons]

    localizations = []
    for row, col in zip(rows, cols):
        x = (col / (w - 1)) * ROOM_X
        y = (row / (h - 1)) * ROOM_Y
        localizations.append([
            float(np.clip(x, 0.0, ROOM_X)),
            float(np.clip(y, 0.0, ROOM_Y)),
        ])
    return localizations, scores.tolist()


def gt_to_localizations(xy: np.ndarray, mask: np.ndarray) -> list[list[float]]:
    localizations = []
    for i in range(len(mask)):
        if mask[i] < 0.5:
            continue
        x, y = xy[i]
        localizations.append([
            float(np.clip(x, 0.0, ROOM_X)),
            float(np.clip(y, 0.0, ROOM_Y)),
        ])
    return localizations


def cost_matrix(gt: list[list[float]], pred: list[list[float]]) -> np.ndarray:
    gt_arr = np.array(gt, dtype=np.float64)
    pred_arr = np.array(pred, dtype=np.float64)
    diff = gt_arr[:, None, :] - pred_arr[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=-1))


def match_hungarian(
    gt: list[list[float]],
    pred: list[list[float]],
    threshold: float = MATCH_THRESHOLD,
) -> tuple[list[float], int, int]:
    n_gt = len(gt)
    n_pred = len(pred)
    if n_gt == 0 and n_pred == 0:
        return [], 0, 0
    if n_gt == 0:
        return [], 0, n_pred
    if n_pred == 0:
        return [], n_gt, 0

    costs = cost_matrix(gt, pred)
    row_i, col_i = linear_sum_assignment(costs)
    matched_gt = set()
    matched_pred = set()
    distances = []

    for row, col in zip(row_i, col_i):
        distance = float(costs[row, col])
        if distance <= threshold:
            distances.append(distance)
            matched_gt.add(row)
            matched_pred.add(col)

    return distances, n_gt - len(matched_gt), n_pred - len(matched_pred)


def evaluate_official_style_from_heatmaps(
    y_pred_heat: np.ndarray,
    y_coordinates: np.ndarray,
    y_mask: np.ndarray,
    threshold: float,
    nms_size: int = NMS_SIZE,
    match_threshold: float = MATCH_THRESHOLD,
    max_persons: int = MAX_PEOPLE,
) -> dict:
    total_tp = 0
    total_fp = 0
    total_fn = 0
    all_tp_distances = []
    count_abs_errors = []
    count_exact = 0

    for frame_id in range(len(y_pred_heat)):
        gt_locs = gt_to_localizations(y_coordinates[frame_id], y_mask[frame_id])
        pred_locs, _ = heatmap_to_localizations(
            y_pred_heat[frame_id],
            threshold=threshold,
            nms_size=nms_size,
            max_persons=max_persons,
        )
        distances, n_fn, n_fp = match_hungarian(gt_locs, pred_locs, threshold=match_threshold)
        total_tp += len(distances)
        total_fp += n_fp
        total_fn += n_fn
        all_tp_distances.extend(distances)
        count_abs_errors.append(abs(len(pred_locs) - len(gt_locs)))
        if len(pred_locs) == len(gt_locs):
            count_exact += 1

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    if all_tp_distances:
        distances = np.array(all_tp_distances, dtype=np.float64)
        rmse = float(np.sqrt(np.mean(distances ** 2)))
        mae = float(np.mean(distances))
        median = float(np.median(distances))
        p90 = float(np.percentile(distances, 90))
        matched_pairs = int(len(distances))
    else:
        rmse = None
        mae = None
        median = None
        p90 = None
        matched_pairs = 0

    return {
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn),
        "rmse": rmse,
        "mae": mae,
        "median": median,
        "p90": p90,
        "matched_pairs": matched_pairs,
        "count_mae": float(np.mean(count_abs_errors)) if count_abs_errors else 0.0,
        "count_accuracy": float(count_exact / len(y_pred_heat)) if len(y_pred_heat) else 0.0,
        "threshold": float(threshold),
        "nms_size": int(nms_size),
        "match_threshold": float(match_threshold),
    }


def sweep_thresholds(
    y_pred_heat: np.ndarray,
    y_coordinates: np.ndarray,
    y_mask: np.ndarray,
    thresholds: np.ndarray,
) -> tuple[list[dict], dict]:
    results = []
    for threshold in thresholds:
        result = evaluate_official_style_from_heatmaps(
            y_pred_heat,
            y_coordinates,
            y_mask,
            threshold=float(threshold),
            nms_size=NMS_SIZE,
            match_threshold=MATCH_THRESHOLD,
            max_persons=MAX_PEOPLE,
        )
        results.append(result)
        print(
            f"threshold={threshold:.2f} | F1={result['f1']:.4f} | "
            f"P={result['precision']:.4f} | R={result['recall']:.4f} | "
            f"Count MAE={result['count_mae']:.3f}"
        )
    return results, max(results, key=lambda row: row["f1"])


def attach_validation_selection_metrics(metrics: dict, validation_best: dict | None) -> dict:
    """Attach validation metrics without overwriting test metrics."""
    if not validation_best:
        return metrics

    enriched = dict(metrics)
    enriched["selection_split"] = "validation"
    enriched["selection_metric"] = "validation_f1"
    enriched["validation_best_f1"] = validation_best["f1"]
    enriched["validation_best_precision"] = validation_best["precision"]
    enriched["validation_best_recall"] = validation_best["recall"]
    enriched["validation_best_threshold"] = validation_best["threshold"]
    enriched["validation_best_nms_size"] = validation_best["nms_size"]
    return enriched


def print_result_row(row: dict) -> None:
    print(
        f"{row['experiment']:>10} | "
        f"val_f1={row.get('validation_best_f1', float('nan')):.4f} | "
        f"val_thr={row.get('validation_best_threshold', float('nan')):.2f} | "
        f"test_f1={row.get('f1', float('nan')):.4f} | "
        f"test_mae={row.get('mae', float('nan')):.3f} | "
        f"count_mae={row.get('count_mae', float('nan')):.3f}"
    )


def maybe_cap_files(files: list[str], cap: int | None) -> list[str]:
    if cap is None or cap <= 0:
        return files
    return files[:cap]


def maybe_cap_samples(
    x: np.ndarray,
    y_coordinates: np.ndarray,
    y_mask: np.ndarray,
    cap: int | None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if cap is None or cap <= 0 or len(x) <= cap:
        return x, y_coordinates, y_mask
    return x[:cap], y_coordinates[:cap], y_mask[:cap]


def run_experiment(
    bin_config: BinConfig,
    base_train_files: list[str],
    base_validation_files: list[str],
    base_test_files: list[str],
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    max_files_per_split: int | None,
    max_train_samples: int | None,
    max_validation_samples: int | None,
    max_test_samples: int | None,
) -> dict:
    set_global_seed(seed)

    train_files = maybe_cap_files(base_train_files, max_files_per_split)
    validation_files = maybe_cap_files(base_validation_files, max_files_per_split)
    test_files = maybe_cap_files(base_test_files, max_files_per_split)

    print("\n" + "=" * 80)
    print(f"Experiment {bin_config.name}: bins [{bin_config.start}:{bin_config.end}]")
    print("=" * 80)

    print("Building train dataset")
    x_train, y_coordinates_train, y_mask_train, train_frame_idx, train_window_name = (
        build_dataset(train_files, bin_config)
    )
    print("Building validation dataset")
    x_validation, y_coordinates_validation, y_mask_validation, validation_frame_idx, validation_window_name = (
        build_dataset(validation_files, bin_config)
    )
    print("Building test dataset")
    x_test, y_coordinates_test, y_mask_test, test_frame_idx, test_window_name = (
        build_dataset(test_files, bin_config)
    )

    y_coordinates_train = sanitize_active_coordinates("train", y_coordinates_train, y_mask_train)
    y_coordinates_validation = sanitize_active_coordinates(
        "validation", y_coordinates_validation, y_mask_validation
    )
    y_coordinates_test = sanitize_active_coordinates("test", y_coordinates_test, y_mask_test)

    validate_dataset_split("train", x_train, y_coordinates_train, y_mask_train, bin_config)
    validate_dataset_split(
        "validation", x_validation, y_coordinates_validation, y_mask_validation, bin_config
    )
    validate_dataset_split("test", x_test, y_coordinates_test, y_mask_test, bin_config)

    x_train, x_validation, x_test, _, _ = normalize_datasets(
        x_train,
        x_validation,
        x_test,
    )

    x_train, y_coordinates_train, y_mask_train = maybe_cap_samples(
        x_train,
        y_coordinates_train,
        y_mask_train,
        max_train_samples,
    )
    x_validation, y_coordinates_validation, y_mask_validation = maybe_cap_samples(
        x_validation,
        y_coordinates_validation,
        y_mask_validation,
        max_validation_samples,
    )
    x_test, y_coordinates_test, y_mask_test = maybe_cap_samples(
        x_test,
        y_coordinates_test,
        y_mask_test,
        max_test_samples,
    )

    y_heat_train = build_heatmaps(y_coordinates_train, y_mask_train)
    y_heat_validation = build_heatmaps(y_coordinates_validation, y_mask_validation)

    tf.keras.backend.clear_session()
    set_global_seed(seed)

    model = build_radar_heatmap_unet(
        input_shape=x_train.shape[1:],
        heatmap_h=HEATMAP_H,
        heatmap_w=HEATMAP_W,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=stable_focal_dice_mass_loss,
        metrics=["mae"],
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            mode="min",
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            mode="min",
            verbose=1,
        ),
    ]

    history = model.fit(
        x_train,
        y_heat_train,
        validation_data=(x_validation, y_heat_validation),
        batch_size=batch_size,
        epochs=epochs,
        shuffle=True,
        callbacks=callbacks,
    )

    y_pred_validation = model.predict(x_validation, batch_size=batch_size)
    sweep_results, best_result = sweep_thresholds(
        y_pred_validation,
        y_coordinates_validation,
        y_mask_validation,
        thresholds=np.arange(0.05, 0.99, 0.05),
    )

    y_pred_test = model.predict(x_test, batch_size=batch_size)
    final_results = evaluate_official_style_from_heatmaps(
        y_pred_test,
        y_coordinates_test,
        y_mask_test,
        threshold=best_result["threshold"],
        nms_size=NMS_SIZE,
        match_threshold=MATCH_THRESHOLD,
        max_persons=MAX_PEOPLE,
    )

    metrics = {
        "experiment": bin_config.name,
        "bin_start": bin_config.start,
        "bin_end": bin_config.end,
        "num_bins": bin_config.width,
        "range_start_m": bin_config.start * 0.15,
        "range_end_m": bin_config.end * 0.15,
        "epochs_requested": epochs,
        "epochs_trained": len(history.history.get("loss", [])),
        "train_samples": int(len(x_train)),
        "validation_samples": int(len(x_validation)),
        "test_samples": int(len(x_test)),
        **final_results,
    }
    metrics = attach_validation_selection_metrics(metrics, best_result)

    tf.keras.backend.clear_session()
    del model
    gc.collect()

    return metrics

# Run Ablation


In [6]:
hf_token = HF_TOKEN or os.environ.get('HF_TOKEN')
npz_files = find_dataset(PROJECT_DIR, DATASET_REPO, hf_token=hf_token)
train_files, validation_files, test_files = make_file_splits(npz_files, RANDOM_STATE)

summary_rows = []
for bin_config in BIN_CONFIGS:
    metrics = run_experiment(
        bin_config=bin_config,
        base_train_files=train_files,
        base_validation_files=validation_files,
        base_test_files=test_files,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        seed=RANDOM_STATE,
        max_files_per_split=MAX_FILES_PER_SPLIT,
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_validation_samples=MAX_VALIDATION_SAMPLES,
        max_test_samples=MAX_TEST_SAMPLES,
    )
    summary_rows.append(metrics)
    print_result_row(metrics)

best = max(summary_rows, key=lambda row: row.get('validation_best_f1', float('-inf')))
print('\\nBest bin configuration by validation F1:')
print_result_row(best)


Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Dataset path: /root/.cache/huggingface/hub/datasets--HAEEAI--multi-person-localization/snapshots/265b3e54e10e109122a2539e400abfeb6d5f05df
Total .npz files: 24
Train: 15 windows | 0p=1, 1p=4, 2p=3, 3p=2, 4p=5
Validation: 4 windows | 0p=0, 1p=1, 2p=1, 3p=1, 4p=1
Test: 5 windows | 0p=1, 1p=1, 2p=1, 3p=1, 4p=1

Experiment bins_4_50: bins [4:50]
Building train dataset
  [1/15] window_000000.npz
  [2/15] window_000001.npz
  [3/15] window_000003.npz
  [4/15] window_000005.npz
  [5/15] window_000006.npz
  [6/15] window_000007.npz
  [7/15] window_000008.npz
  [8/15] window_000009.npz
  [9/15] window_000012.npz
  [10/15] window_000013.npz
  [11/15] window_000017.npz
  [12/15] window_000018.npz
  [13/15] window_000019.npz
  [14/15] window_000020.npz
  [15/15] window_000023.npz
Building validation dataset
  [1/4] window_000004.npz
  [2/4] window_000010.npz
  [3/4] window_000014.npz
  [4/4] window_000021.npz
Building test dataset
  [1/5] window_000002.npz
  [2/5] window_000011.npz
  [3/5] window_00

KeyboardInterrupt: 